# demo_rl

> Phase 12, Track B on this Mac (`specs/multiagent/phase12/`): a debug run of the training loop with a 0.5B model, the warm-start check of the 3B model trained in Colab, the evaluation of the trained models (with Claude Haiku teammates for cross-play) and the report. The GPU runs themselves are in `colab_clue_rl`.

Artifacts under `results/multiagent/rl/{quick,debug,warm,eval}/` (gitignored); the Colab zip is unpacked into `results/multiagent/rl/colab/`; local replies are cached in `results/multiagent/rl/cache/`, so a rerun of the evaluation is byte-identical. GPU hours and cost per Colab session go in `results/multiagent/rl/gpu.json` (`{"hours": ..., "cost": ...}`).

- **Quick** (normal tests): the evaluation and report pipeline with scripted policies.
- **Debug**, **warm-start check** and **evaluation** (`--cell-timeout 0 --flags train`; the evaluation's cross-play cells also need `EXPLRL_LLM_LIVE=1`): these load models (the 0.5B and 3B downloads are about 1 GB and 6 GB).

In [ ]:
import json
import os
import subprocess

import numpy as np

from explRL.bench_grid import readCSV, rowKey, writeCSV
from explRL.clue_policy import claudePolicy, scriptedPolicy
from explRL.clue_roll import runEval
from explRL.coop_grid import LOCAL, MODELS, RL_PLAN, coopHash, evalSuite
from explRL.llm_budget import Budget, estimateCost
from explRL.llm_grid import HAIKU
from explRL.rl_report import rlSummaries, writeRL, writeRLFigures
from explRL.train import demo_artifact_dir

DN = 'multiagent/rl'
D = demo_artifact_dir(None, DN)
CAP = 150.0
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=CAP, ledger=str(LLM / 'spend.jsonl'))
P11 = demo_artifact_dir('full', 'multiagent/clue') / 'runs.csv'


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def slots(cell, model, haiku):
    "(policies, memory) for a cell: model agents with notes; in cross-play, Haiku slots with full memory."
    test, N = cell['level'].split('|')[1], cell['N']
    mine = {'cross1': [0], 'cross3': [0, 1, 2]}.get(test)
    if mine is None:
        return model, 'notes'
    return {i: model if i in mine else haiku for i in range(N)}, {i: 'notes' if i in mine else 'full' for i in range(N)}


def report(mode, rows, apiCost=0.0, adapters=None, curves=None):
    out = demo_artifact_dir(mode, DN)
    writeCSV(rows, out / 'runs.csv')
    p11 = readCSV(P11) if P11.exists() else []
    S = rlSummaries(rows, [r for r in p11 if r['runner'] == 'llm'], [r for r in p11 if r['study'] == 'clueNumpy'])
    gpu = json.loads((D / 'gpu.json').read_text()) if (D / 'gpu.json').exists() else dict(hours=0.0, cost=0.0)
    meta = dict(mode=mode, planHash=coopHash(RL_PLAN), commit=gitCommit(), base=MODELS['base'], adapters=adapters or {},
                gpuHours=gpu['hours'], gpuCost=gpu['cost'], apiCost=apiCost,
                apiSpent=Budget(CAP, LIVE_SPEC['ledger']).spent() if (LLM / 'spend.jsonl').exists() else 0.0)
    writeRL(out / 'report.md', S, meta)
    hk = S['main'] and [r['welfare'] for r in p11 if r.get('study') == 'clue' and r['channel'] == 'blackboard' and int(r['N']) == 4]
    writeRLFigures(out, S, curves, refs=[(float(np.mean(hk)), 'Haiku (Phase 11)', 'tab:orange')] if hk else ())
    return S

## Quick: the evaluation and report pipeline with scripted policies

In [ ]:
import re
def solver(system, blocks):
    "Reads only the text: opens the first listed subtask, submits once its known clues fix the answer, else waits."
    o = blocks[-1]['text']
    m = re.search(r'Your open subtask: T(\d+), open\. Find x with 0 <= x < (\d+)\. Clues you know: (.*)\.', o)
    if not m:
        return json.dumps(dict(action='open', target=re.search(r'\): (T\d+)', o).group(1), answer=0, clue='', notes=''))
    cl = [tuple(map(int, c)) for c in re.findall(r'x mod (\d+) = (\d+)', m.group(3))]
    xs = [y for y in range(int(m.group(2))) if all(y % p == r for p, r in cl)]
    return json.dumps(dict(action='submit', answer=xs[0], target='', clue='', notes='') if len(xs) == 1 else dict(action='wait'))
waiter = scriptedPolicy(lambda s, b: '{"action": "wait"}')
cells = [dict(c, cfg=dict(c['cfg'], turns=3)) for c in evalSuite(('warm', 'credit')) if c['seed'] in (900, 901, 930)]
rows = runEval(cells, lambda c: slots(c, scriptedPolicy(solver), waiter))
S = report('quick', rows)
q = demo_artifact_dir('quick', DN)
assert {p.name for p in q.iterdir()} >= {'runs.csv', 'report.md', 'coop_main.png', 'coop_info.png'} and 'credit' in S['gain']
print(open(q / 'report.md').read()[:600])

## Debug (`train`): the whole training loop with the 0.5B model on this Mac (AC-6)

50 warm-start steps, merge, the warm-start check on 3 boards (no pass required at 0.5B), two RL steps, resume at step 3, and refusal of a checkpoint written by another configuration.

In [ ]:
#| train
from dataclasses import asdict
from explRL.clue_rl import RLConfig, trainRL
from explRL.clue_sft import SFTConfig, mergeAdapter, sftExamples, trainSFT, warmCheck
from explRL.local_policy import LocalPolicy
G = demo_artifact_dir('debug', DN)
adapter = trainSFT(SFTConfig(MODELS['debug'], str(G / 'sft'), maxSteps=50), sftExamples(range(20000, 20010), turns=20))
warm = mergeAdapter(MODELS['debug'], adapter, G / 'warm')
print('warm-start check (0.5B):', warmCheck(LocalPolicy(warm), seeds=range(1200, 1203)))
cfg = RLConfig(warm, str(G / 'rl'), steps=2, seedsPerStep=2, group=2, turns=4, ckptEvery=1)
trainRL(cfg)
trainRL(RLConfig(**{**asdict(cfg), 'steps': 3}))
m = [json.loads(x) for x in open(G / 'rl' / 'metrics.jsonl')]
assert [x['step'] for x in m] == [1, 2, 3] and all(np.isfinite(x['loss']) for x in m), m
try:
    trainRL(RLConfig(**{**asdict(cfg), 'lr': 2e-5, 'steps': 4}))
    raise AssertionError('a checkpoint of another configuration must be refused')
except ValueError:
    pass
print(m)

## Warm-start check (`train`): the 3B warm-start model from Colab (AC-9)

Unpack the Colab zip into `results/multiagent/rl/colab/` first. The warm-start model is rebuilt here by merging the small adapter into the base model (needed anyway as the untrained baseline).

In [ ]:
#| train
from explRL.clue_sft import mergeAdapter, warmCheck
from explRL.local_policy import LocalPolicy
C, WARM = D / 'colab', D / 'warm_model'
if not (WARM / 'config.json').exists():
    mergeAdapter(MODELS['base'], C / 'warm' / 'adapter', WARM)
res = warmCheck(LocalPolicy(str(WARM), cache=str(D / 'cache')), seeds=range(*[RL_PLAN['warmCheck']['seeds'][0], RL_PLAN['warmCheck']['seeds'][1] + 1]),
                turns=RL_PLAN['warmCheck']['turns'])
(demo_artifact_dir('warm', DN) / 'check.json').write_text(json.dumps(res, indent=1) + '\n')
print(res, '-> passes' if res['passed'] else '-> FAILS: stop and decide (P12-2: the 7-8B check)')

## Evaluation and report (`train`; cross-play also needs `EXPLRL_LLM_LIVE=1`) (AC-11)

The five models one at a time (the untrained base model, the warm-start model, and the three RL adapters on it), cached replies; Haiku slots through the Phase 10 client under the $150 API cap.

In [ ]:
#| train
import gc
import torch
from explRL.clue_policy import claudePolicy
from explRL.llm_runs import LLMConfig, makeClient
from explRL.local_policy import LocalPolicy, adapterHash
assert RL_PLAN['frozen'], 'RL_PLAN must be frozen before the evaluation'
C, WARM = D / 'colab', D / 'warm_model'
cells = evalSuite()
haiku, apiCost = None, 0.0
if LIVE:
    client = makeClient(LLMConfig(task='clue', model=HAIKU, effort=RL_PLAN['haiku']['effort'],
                                  maxTokens=RL_PLAN['haiku']['maxTokens']), LIVE_SPEC)
    client.reserve(estimateCost(HAIKU, 'clue', 1, 3200))
    haiku = claudePolicy(client)
else:
    cells = [c for c in cells if not c['level'].endswith(('cross1', 'cross3'))]
    print('cross-play skipped: set EXPLRL_LLM_LIVE=1')
adapters, rows = {}, []
for m in LOCAL:
    path = None if m in ('untrained', 'warm') else str(C / 'runs' / m / 'final')
    pol = LocalPolicy(MODELS['base'] if m == 'untrained' else str(WARM), adapter=path, cache=str(D / 'cache'))
    adapters[m] = adapterHash(path)
    rows += runEval([c for c in cells if c['level'].startswith(m + '|')], lambda c: slots(c, pol, haiku),
                    logDir=demo_artifact_dir('eval', DN) / 'turns')
    del pol
    gc.collect()
    torch.mps.empty_cache() if torch.backends.mps.is_available() else None
apiCost = float(sum(r['cost'] for r in rows))
curves = {sc: [json.loads(x) for x in open(C / 'runs' / sc / 'metrics.jsonl')] for sc in RL_PLAN['schemes']
          if (C / 'runs' / sc / 'metrics.jsonl').exists()}
S = report('eval', sorted(rows, key=rowKey), apiCost, adapters, curves)
print(open(demo_artifact_dir('eval', DN) / 'report.md').read()[:3000])